# FEM FEniCSx — Kim 2024 Caso B
**6 cables XLPE 154 kV — suelo multicapa + zona PAC**

Modelo FEM completo sin simplificaciones:
- 6 cables con 4 capas radiales cada uno (conductor Cu, XLPE, pantalla, cubierta)
- 3 bandas de suelo (Kim 2024, Tabla 1)
- Zona PAC rectangular (k = 2.094 W/mK)
- BC Robin en superficie, Dirichlet en fondo y laterales
- Fuente volumétrica exacta en el conductor (sin aproximación Gaussiana)

Referencia: Kim et al. (2024). *Geothermics* 125, 103151. DOI: 10.1016/j.geothermics.2024.103151

**Resultado esperado:** T_max_conductor ≈ 70.6 °C

## Celda 1 — Instalación
Ejecutar **una sola vez**. Después **reiniciar el runtime** (`Runtime → Restart runtime`) antes de continuar.

In [ ]:
import subprocess

# Librerías del sistema requeridas por gmsh (OpenGL/GLU)
subprocess.run([
    "apt-get", "install", "-y", "-q",
    "libglu1-mesa", "libxrender1", "libxcursor1", "libxft2", "libxinerama1"
], check=True)

# FEniCSx
try:
    import dolfinx
    print(f"dolfinx ya instalado: {dolfinx.__version__}")
except ImportError:
    !wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh" -O /tmp/fenicsx-install.sh
    !bash /tmp/fenicsx-install.sh

# gmsh
try:
    import gmsh
    print("gmsh ya instalado")
except ImportError:
    !wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh" -O /tmp/gmsh-install.sh
    !bash /tmp/gmsh-install.sh

print("\n✓ Instalación completa. REINICIA EL RUNTIME antes de continuar.")
print("  Runtime → Restart runtime")

## Celda 2 — Imports y verificación
Ejecutar tras reiniciar el runtime.

In [ ]:
import math, sys, time, warnings
import numpy as np

import gmsh
import meshio
import dolfinx
import basix.ufl
from dolfinx import mesh as dmesh, fem, io
from dolfinx.mesh import create_mesh, meshtags, locate_entities_boundary
from dolfinx.fem import (
    functionspace, Function, Constant, dirichletbc, locate_dofs_topological,
)
from dolfinx.fem.petsc import LinearProblem
from mpi4py import MPI
import ufl
from petsc4py import PETSc

print(f"dolfinx:  {dolfinx.__version__}")
print(f"gmsh:     {gmsh.__version__}")
print("Imports OK")

## Celda 3 — Parámetros del problema (Kim 2024, Caso B)

In [ ]:
# --- Dominio [m] ---
XMIN, XMAX = -45.5, 45.5
YMIN, YMAX = -45.5, 0.0

# --- Condiciones de frontera ---
T_BOT  = 288.35   # K  Dirichlet inferior/laterales
T_AIR  = 300.35   # K  Robin superficie
H_CONV = 7.371    # W/(m²·K)

# --- Posiciones de cables (cables_placement.csv) ---
CABLES = [
    (-0.40, -1.60), ( 0.00, -1.60), ( 0.40, -1.60),
    (-0.40, -1.20), ( 0.00, -1.20), ( 0.40, -1.20),
]
N_CABLES = len(CABLES)

# --- Capas del cable (cable_layers.csv): (r_inner, r_outer, k, nombre) ---
CABLE_LAYERS = [
    (0.00000, 0.01885, 400.0,    "conductor_Cu"),
    (0.01885, 0.04085,   0.2857, "XLPE"),
    (0.04085, 0.04935, 384.6,   "pantalla_metal"),
    (0.04935, 0.05335,   0.45,  "cubierta_PE"),
]
N_LAYERS    = len(CABLE_LAYERS)
R_CONDUCTOR = CABLE_LAYERS[0][1]   # 0.01885 m
R_CABLE     = CABLE_LAYERS[-1][1]  # 0.05335 m

# --- Suelo 3 bandas (Kim 2024): (y_top, y_bottom, k) ---
SOIL_BANDS = [
    ( 0.000,  -0.56,  1.804),
    (-0.560,  -1.76,  1.351),
    (-1.760, -45.50,  1.517),
]

# --- Zona PAC (physics_params.csv) ---
PAC_CX, PAC_CY = 0.0, -1.40
PAC_W,  PAC_H  = 1.30,  0.90
PAC_K          = 2.094

# --- Calor lineal por cable (IEC 60287, Kim 2024) ---
# Red coreana → 60 Hz; 1200 mm² Cu, I=1026 A, T_op=70.6°C
# W_d = 3.57 W/m = pérdidas dieléctricas XLPE (valor Kim 2024, run_optim_B.py)
R_DC20_CU = 1.51e-5   # Ω/m
ALPHA_CU  = 0.00393
CURRENT_A = 1026.0
T_OP_K    = 273.15 + 70.6   # K  (temperatura de operación del conductor)
FREQ      = 60.0              # Hz  (red coreana)
W_D_LIN   = 3.57              # W/m por cable (pérdidas dieléctricas XLPE)

def compute_Q_lin(I, R_dc20, alpha_R, T_op_K, T_ref_K=293.15, freq=60.0):
    R_dc_T = R_dc20 * (1.0 + alpha_R * (T_op_K - T_ref_K))
    xs_sq  = 8.0 * math.pi * freq / (R_dc_T * 1.0e7)
    xs_4   = xs_sq ** 2
    ys     = xs_4 / (192.0 + 0.8 * xs_4)
    return I**2 * R_dc_T * (1.0 + ys)

Q_COND = compute_Q_lin(CURRENT_A, R_DC20_CU, ALPHA_CU, T_OP_K, freq=FREQ)
Q_LIN  = Q_COND + W_D_LIN          # total = conductor + dieléctrico

# Fuente volumétrica en el conductor Cu
A_COND  = math.pi * CABLE_LAYERS[0][1] ** 2
Q_VOL   = Q_COND / A_COND

# Fuente volumétrica en el aislamiento XLPE (pérdidas dieléctricas)
A_XLPE  = math.pi * (CABLE_LAYERS[1][1] ** 2 - CABLE_LAYERS[1][0] ** 2)
Q_VOL_D = W_D_LIN / A_XLPE

print(f"Q_cond = {Q_COND:.3f} W/m  (conductor + skin effect)")
print(f"W_d    = {W_D_LIN:.3f} W/m  (dieléctrico XLPE)")
print(f"Q_lin  = {Q_LIN:.3f} W/m  (total)")
print(f"Q_VOL  = {Q_VOL:.1f} W/m³  (conductor)")
print(f"Q_VOL_D= {Q_VOL_D:.1f} W/m³  (XLPE dieléctrico)")
print(f"R_cond = {CABLE_LAYERS[0][1]*1e3:.2f} mm,  R_cable = {R_CABLE*1e3:.2f} mm")


## Celda 4 — Mallado con Gmsh
Malla adaptativa: densa en cables (~3 mm en conductor), más gruesa en suelo lejano (~3.5 m).

In [ ]:
def build_mesh(h_cond=0.003, h_cable=0.015, h_pac=0.08, h_near=0.40, h_far=3.5, verbose=False):
    """Construye la malla con Gmsh (kernel OCC) y la importa en DOLFINx."""
    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 1 if verbose else 0)
    gmsh.model.add("kim2024_caseB")
    occ = gmsh.model.occ

    # Bandas de suelo como superficies explícitas.
    soil_surfs = []
    for y_top, y_bottom, _k in SOIL_BANDS:
        soil_surfs.append(
            occ.addRectangle(XMIN, y_bottom, 0, XMAX - XMIN, y_top - y_bottom)
        )

    # Zona PAC
    px0 = PAC_CX - PAC_W / 2
    py0 = PAC_CY - PAC_H / 2
    pac_surf = occ.addRectangle(px0, py0, 0, PAC_W, PAC_H)

    # Cables por capas y puntos de refinamiento
    cable_disks = []
    refinement_points = []
    for cx, cy in CABLES:
        refinement_points.append(occ.addPoint(cx, cy, 0))
        refinement_points.append(occ.addPoint(cx + R_CONDUCTOR, cy, 0))
        refinement_points.append(occ.addPoint(cx - R_CONDUCTOR, cy, 0))
        refinement_points.append(occ.addPoint(cx, cy + R_CONDUCTOR, 0))
        refinement_points.append(occ.addPoint(cx, cy - R_CONDUCTOR, 0))
        refinement_points.append(occ.addPoint(cx + R_CABLE, cy, 0))
        refinement_points.append(occ.addPoint(cx - R_CABLE, cy, 0))
        refinement_points.append(occ.addPoint(cx, cy + R_CABLE, 0))
        refinement_points.append(occ.addPoint(cx, cy - R_CABLE, 0))

        disks_i = []
        for _j, (_r_in, r_out, _kv, _lname) in enumerate(CABLE_LAYERS):
            disks_i.append(occ.addDisk(cx, cy, 0, r_out, r_out))
        cable_disks.append(disks_i)

    # Puntos para refinamiento de la zona PAC
    pac_ref_points = [
        occ.addPoint(px0, py0, 0),
        occ.addPoint(px0 + PAC_W, py0, 0),
        occ.addPoint(px0, py0 + PAC_H, 0),
        occ.addPoint(px0 + PAC_W, py0 + PAC_H, 0),
        occ.addPoint(PAC_CX, PAC_CY, 0),
    ]

    occ.synchronize()

    # Fragmentar para imponer interfaces compartidas.
    object_surfs = [(2, s) for s in soil_surfs]
    object_surfs += [(2, pac_surf)]
    object_surfs += [(2, s) for disks_i in cable_disks for s in disks_i]
    occ.fragment(object_surfs, [])
    occ.synchronize()

    # Refinamiento de malla con campos de tamaño alrededor de cables y PAC.
    cable_distance = gmsh.model.mesh.field.add("Distance")
    gmsh.model.mesh.field.setNumbers(cable_distance, "PointsList", refinement_points)
    gmsh.model.mesh.field.setNumber(cable_distance, "Sampling", 100)

    cable_threshold = gmsh.model.mesh.field.add("Threshold")
    gmsh.model.mesh.field.setNumber(cable_threshold, "InField", cable_distance)
    gmsh.model.mesh.field.setNumber(cable_threshold, "SizeMin", h_cond)
    gmsh.model.mesh.field.setNumber(cable_threshold, "SizeMax", h_near)
    gmsh.model.mesh.field.setNumber(cable_threshold, "DistMin", R_CONDUCTOR)
    gmsh.model.mesh.field.setNumber(cable_threshold, "DistMax", 0.8)

    pac_distance = gmsh.model.mesh.field.add("Distance")
    gmsh.model.mesh.field.setNumbers(pac_distance, "PointsList", pac_ref_points)
    gmsh.model.mesh.field.setNumber(pac_distance, "Sampling", 50)

    pac_threshold = gmsh.model.mesh.field.add("Threshold")
    gmsh.model.mesh.field.setNumber(pac_threshold, "InField", pac_distance)
    gmsh.model.mesh.field.setNumber(pac_threshold, "SizeMin", h_pac)
    gmsh.model.mesh.field.setNumber(pac_threshold, "SizeMax", h_far)
    gmsh.model.mesh.field.setNumber(pac_threshold, "DistMin", 0.2)
    gmsh.model.mesh.field.setNumber(pac_threshold, "DistMax", 2.0)

    min_field = gmsh.model.mesh.field.add("Min")
    gmsh.model.mesh.field.setNumbers(min_field, "FieldsList", [cable_threshold, pac_threshold])
    gmsh.model.mesh.field.setAsBackgroundMesh(min_field)
    gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
    gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
    gmsh.option.setNumber("Mesh.MeshSizeMin", h_cond)
    gmsh.option.setNumber("Mesh.MeshSizeMax", h_far)

    # Grupo físico para TODAS las superficies 2D (necesario para que Gmsh
    # exporte los triángulos al .msh — sin esto meshio no los ve).
    all_surfs = [tag for dim, tag in gmsh.model.getEntities(2)]
    gmsh.model.addPhysicalGroup(2, all_surfs, 99)
    gmsh.model.setPhysicalName(2, 99, "domain")

    # Fronteras del dominio exterior
    top_tags, bot_tags, left_tags, right_tags = [], [], [], []
    tol = 0.5
    for dim, tag in gmsh.model.getEntities(1):
        xlo, ylo, _, xhi, yhi, _ = gmsh.model.getBoundingBox(dim, tag)
        ymid = 0.5 * (ylo + yhi)
        xmid = 0.5 * (xlo + xhi)
        if abs(ymid - YMAX) < tol and (xhi - xlo) > tol:
            top_tags.append(tag)
        elif abs(ymid - YMIN) < tol and (xhi - xlo) > tol:
            bot_tags.append(tag)
        elif abs(xmid - XMIN) < tol and (yhi - ylo) > tol:
            left_tags.append(tag)
        elif abs(xmid - XMAX) < tol and (yhi - ylo) > tol:
            right_tags.append(tag)

    for tags, pid, name in [
        (top_tags, 10, "top"),
        (bot_tags, 11, "bot"),
        (left_tags, 12, "left"),
        (right_tags, 13, "right"),
    ]:
        if tags:
            gmsh.model.addPhysicalGroup(1, tags, pid)
            gmsh.model.setPhysicalName(1, pid, name)

    gmsh.model.mesh.generate(2)
    gmsh.model.mesh.optimize("Netgen")
    gmsh.write("/tmp/kim2024.msh")
    gmsh.finalize()

    raw = meshio.read("/tmp/kim2024.msh")
    pts = raw.points[:, :2]
    tri_cells = raw.get_cells_type("triangle").astype(np.int64)
    print(f"  meshio: {len(pts)} nodos, {len(tri_cells)} triángulos")

    domain = ufl.Mesh(basix.ufl.element("Lagrange", "triangle", 1, shape=(2,)))
    msh = create_mesh(MPI.COMM_WORLD, tri_cells, domain, pts)
    n_cells = msh.topology.index_map(msh.topology.dim).size_global
    n_dofs = functionspace(msh, ("Lagrange", 1)).dofmap.index_map.size_global
    print(f"Malla OK: {n_cells:,d} celdas, {n_dofs:,d} DOFs")

    # Reetiquetar celdas sobre la malla importada para evitar errores si create_mesh reordena celdas.
    cell_vertices = np.asarray(msh.geometry.dofmap, dtype=np.int64)
    cell_centers = msh.geometry.x[cell_vertices].mean(axis=1)
    centers_x = cell_centers[:, 0]
    centers_y = cell_centers[:, 1]
    cell_values = np.full(len(cell_vertices), 3, dtype=np.int32)

    for idx, (cx, cy) in enumerate(CABLES):
        r = np.sqrt((centers_x - cx) ** 2 + (centers_y - cy) ** 2)
        cell_values[r <= CABLE_LAYERS[0][1]] = 100 * (idx + 1) + 0
        cell_values[(r > CABLE_LAYERS[1][0]) & (r <= CABLE_LAYERS[1][1])] = 100 * (idx + 1) + 1
        cell_values[(r > CABLE_LAYERS[2][0]) & (r <= CABLE_LAYERS[2][1])] = 100 * (idx + 1) + 2
        cell_values[(r > CABLE_LAYERS[3][0]) & (r <= CABLE_LAYERS[3][1])] = 100 * (idx + 1) + 3

    pac_mask = (
        (centers_x >= px0)
        & (centers_x <= px0 + PAC_W)
        & (centers_y >= py0)
        & (centers_y <= py0 + PAC_H)
    )
    non_cable_mask = cell_values < 100
    cell_values[pac_mask & non_cable_mask] = 4
    cell_values[(centers_y > SOIL_BANDS[0][1]) & (centers_y <= SOIL_BANDS[0][0]) & non_cable_mask & ~pac_mask] = 1
    cell_values[(centers_y > SOIL_BANDS[1][1]) & (centers_y <= SOIL_BANDS[1][0]) & non_cable_mask & ~pac_mask] = 2
    cell_values[(centers_y > SOIL_BANDS[2][1]) & (centers_y <= SOIL_BANDS[2][0]) & non_cable_mask & ~pac_mask] = 3

    cell_tags = meshtags(
        msh,
        msh.topology.dim,
        np.arange(len(cell_values), dtype=np.int32),
        cell_values,
    )

    unique_tags, unique_counts = np.unique(cell_values, return_counts=True)
    print("Tags de celda:", {int(tag): int(count) for tag, count in zip(unique_tags, unique_counts)})

    msh.topology.create_connectivity(1, 2)
    tol = 1e-6
    top_f = locate_entities_boundary(msh, 1, lambda x: np.isclose(x[1], YMAX, atol=tol))
    bot_f = locate_entities_boundary(msh, 1, lambda x: np.isclose(x[1], YMIN, atol=tol))
    left_f = locate_entities_boundary(msh, 1, lambda x: np.isclose(x[0], XMIN, atol=tol))
    right_f = locate_entities_boundary(msh, 1, lambda x: np.isclose(x[0], XMAX, atol=tol))

    f_idx = np.concatenate([top_f, bot_f, left_f, right_f])
    f_val = np.concatenate([
        np.full(len(top_f), 10, np.int32),
        np.full(len(bot_f), 11, np.int32),
        np.full(len(left_f), 12, np.int32),
        np.full(len(right_f), 13, np.int32),
    ])
    srt = np.argsort(f_idx)
    facet_tags = meshtags(msh, 1, f_idx[srt], f_val[srt])

    print(
        "Fronteras:",
        {"top": len(top_f), "bot": len(bot_f), "left": len(left_f), "right": len(right_f)},
    )

    return msh, cell_tags, facet_tags

msh, cell_tags, facet_tags = build_mesh()


## Celda 5 — Resolver FEM (MUMPS)

In [ ]:
def build_k_function(msh, cell_tags):
    V0 = functionspace(msh, ("DG", 0))
    k_func = Function(V0)
    k_map = {
        1: SOIL_BANDS[0][2], 2: SOIL_BANDS[1][2], 3: SOIL_BANDS[2][2],
        4: PAC_K,
    }
    for i in range(N_CABLES):
        for j, (ri, ro, kv, nm) in enumerate(CABLE_LAYERS):
            k_map[100*(i+1)+j] = kv
    k_func.x.array[:] = SOIL_BANDS[2][2]
    for tag, kv in k_map.items():
        cells = cell_tags.indices[cell_tags.values == tag]
        k_func.x.array[cells] = kv
    k_func.x.scatter_forward()
    return k_func


def build_Q_function(msh, cell_tags):
    """Asigna fuentes de calor volumétricas.

    Solo pérdidas Joule + skin effect en el conductor Cu (IEC 60287, 60 Hz).

    NOTA: Las pérdidas dieléctricas W_d NO se depositan como fuente volumétrica
    en el XLPE. En el circuito térmico IEC 60287, W_d sólo ve las resistencias
    EXTERNAS al cable (T2, T3, T4) — no la resistencia de aislamiento T1. Si se
    deposita W_d en el volumen de XLPE, el calor fluye hacia adentro a través del
    conductor (resistencia ≈ 0) y luego hacia afuera a través de toda la capa XLPE,
    añadiendo T1 al camino térmico de W_d y sobreestimando la temperatura del
    conductor en varios grados. El modelo COMSOL de Kim 2024 produce 70.6 °C
    usando sólo Q_cond (60 Hz) en el conductor.
    """
    V0 = functionspace(msh, ("DG", 0))
    Q_func = Function(V0)
    Q_func.x.array[:] = 0.0
    for i in range(N_CABLES):
        cells_c = cell_tags.indices[cell_tags.values == 100*(i+1)+0]
        Q_func.x.array[cells_c] = Q_VOL
    Q_func.x.scatter_forward()
    return Q_func


def solve_thermal(msh, cell_tags, facet_tags):
    V = functionspace(msh, ("Lagrange", 1))
    k_func = build_k_function(msh, cell_tags)
    Q_func = build_Q_function(msh, cell_tags)

    dx = ufl.Measure("dx", domain=msh, subdomain_data=cell_tags)
    ds = ufl.Measure("ds", domain=msh, subdomain_data=facet_tags)

    T_dir = fem.Constant(msh, PETSc.ScalarType(T_BOT))
    bcs = [
        dirichletbc(T_dir, locate_dofs_topological(V, 1, facet_tags.find(11)), V),
        dirichletbc(T_dir, locate_dofs_topological(V, 1, facet_tags.find(12)), V),
        dirichletbc(T_dir, locate_dofs_topological(V, 1, facet_tags.find(13)), V),
    ]

    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    h_c   = fem.Constant(msh, PETSc.ScalarType(H_CONV))
    T_air = fem.Constant(msh, PETSc.ScalarType(T_AIR))

    a = (k_func * ufl.inner(ufl.grad(u), ufl.grad(v)) * dx
         + h_c * u * v * ds(10))
    L = (Q_func * v * dx
         + h_c * T_air * v * ds(10))

    problem = LinearProblem(
        a, L, bcs=bcs,
        petsc_options_prefix="kim2024_thermal_",
        petsc_options={"ksp_type": "preonly", "pc_type": "lu",
                       "pc_factor_mat_solver_type": "mumps"},
    )
    return problem.solve()


print("Resolviendo FEM...")
t1 = time.time()
T_fem = solve_thermal(msh, cell_tags, facet_tags)
T_C   = T_fem.x.array - 273.15
print(f"T_min = {T_C.min():.2f}°C   T_max = {T_C.max():.2f}°C   ({time.time()-t1:.1f}s)")


## Celda 6 — Post-proceso: T_conductor por cable

In [ ]:
V = T_fem.function_space
# DOFs por celda directamente del dofmap de V (evita locate_dofs_topological con dim=2)
V_dofs_per_cell = np.array(V.dofmap.list, dtype=np.int32)  # shape (n_cells, n_local_dofs)

T_cond = []
for i in range(N_CABLES):
    cells = cell_tags.indices[cell_tags.values == 100*(i+1)+0]
    if len(cells) == 0:
        T_cond.append(float('nan'))
        continue
    dofs = V_dofs_per_cell[cells].ravel()
    T_cond.append(float(np.max(T_fem.x.array[dofs])) - 273.15)

print("T_conductor por cable:")
for i, (Tc, (cx,cy)) in enumerate(zip(T_cond, CABLES)):
    print(f"  Cable {i+1}  ({cx:+.2f},{cy:+.2f})m  →  {Tc:.2f} °C")

T_max = max(t for t in T_cond if not math.isnan(t))
FEM_REF = 70.6;  PINN_64x4 = 70.19;  PINN_128x5 = 70.17

print()
print("─" * 60)
print(f"  {'Método':<36}  {'T_max':>8}  {'vs paper':>8}")
print("─" * 60)
print(f"  {'FEM paper (COMSOL, Kim 2024)':<36}  {FEM_REF:>8.2f}  {'—':>8}")
print(f"  {'FEM FEniCSx (malla completa)':<36}  {T_max:>8.2f}  {T_max-FEM_REF:>+7.2f}K")
print(f"  {'PINN 64×4 (Kennelly bg)':<36}  {PINN_64x4:>8.2f}  {PINN_64x4-FEM_REF:>+7.2f}K")
print(f"  {'PINN 128×5 destilado':<36}  {PINN_128x5:>8.2f}  {PINN_128x5-FEM_REF:>+7.2f}K")
print("─" * 60)


## Celda 7 — Visualización del campo de temperatura

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation

coords   = msh.geometry.x[:, :2]
dofmap   = msh.geometry.dofmap
cells_np = np.array(dofmap).reshape(-1, 3)
T_vals   = T_fem.x.array - 273.15
tri      = Triangulation(coords[:,0], coords[:,1], cells_np)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Vista completa
ax = axes[0]
tc = ax.tripcolor(tri, T_vals, shading="flat", cmap="hot_r")
plt.colorbar(tc, ax=ax, label="T [°C]")
ax.set_title("Campo T — dominio completo")
ax.set_xlabel("x [m]");  ax.set_ylabel("y [m]")
ax.set_aspect("equal")

# Zoom cables
ax = axes[1]
tc2 = ax.tripcolor(tri, T_vals, shading="flat", cmap="hot_r")
plt.colorbar(tc2, ax=ax, label="T [°C]")
ax.set_xlim(-1.5, 1.5);  ax.set_ylim(-2.5, 0.2)
ax.set_title(f"Zoom cables — T_max = {T_max:.2f} °C")
ax.set_xlabel("x [m]");  ax.set_ylabel("y [m]")
for cx, cy in CABLES:
    ax.add_patch(plt.Circle((cx,cy), R_CABLE, fill=False, color="cyan", lw=0.8))
    ax.add_patch(plt.Circle((cx,cy), R_CONDUCTOR, fill=False, color="yellow", lw=0.5))

plt.tight_layout()
plt.savefig("fem_T_field_kim2024B.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: fem_T_field_kim2024B.png")

## Celda 8 — Exportar campo FEM a Drive
Guarda los resultados en formato comprimido `.npz` para comparación espacial con modelos PINN en VS Code.


In [ ]:
import os
from pathlib import Path

EXPORT_DIR = Path("/content/drive/MyDrive/Fenics_tesis")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# --- 1. Campo completo: coordenadas de nodos + temperatura ---
# coords_xy : (N_nodos, 2)  —  x, y en metros
# T_K       : (N_nodos,)    —  temperatura en Kelvin
# T_C       : (N_nodos,)    —  temperatura en Celsius
# cells     : (N_celdas, 3) —  conectividad triangular (índices en coords_xy)
coords_xy = msh.geometry.x[:, :2].copy()
T_K_arr   = T_fem.x.array.copy()
T_C_arr   = T_K_arr - 273.15
cells_arr = np.array(msh.geometry.dofmap, dtype=np.int32).reshape(-1, 3)

npz_path = EXPORT_DIR / "fem_field_kim2024B.npz"
np.savez_compressed(
    npz_path,
    coords_xy = coords_xy,   # nodos (x, y) [m]
    T_K       = T_K_arr,     # temperatura en K en cada nodo
    T_C       = T_C_arr,     # temperatura en °C en cada nodo
    cells     = cells_arr,   # triángulos: (n_celdas, 3) índices de nodo
    # Metadatos del problema para cargar en VS Code sin reejecutar
    cables    = np.array(CABLES),
    R_conductor = np.float64(R_CONDUCTOR),
    R_cable     = np.float64(R_CABLE),
    domain    = np.array([XMIN, XMAX, YMIN, YMAX]),
)
print(f"✓ Campo FEM guardado: {npz_path}")
print(f"  coords_xy : {coords_xy.shape}  (nodos)")
print(f"  T_C       : {T_C_arr.shape}   T ∈ [{T_C_arr.min():.2f}, {T_C_arr.max():.2f}] °C")
print(f"  cells     : {cells_arr.shape}  (triángulos)")

# --- 2. Tabla de T_conductor por cable (CSV) ---
import csv
csv_path = EXPORT_DIR / "fem_T_conductor_kim2024B.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["cable", "cx_m", "cy_m", "T_conductor_C", "error_vs_paper_K"])
    for i, (Tc, (cx, cy)) in enumerate(zip(T_cond, CABLES)):
        w.writerow([i+1, cx, cy, f"{Tc:.4f}", f"{Tc - 70.6:.4f}"])
print(f"✓ Tabla conductores: {csv_path}")

# --- 3. Tabla de comparación de métodos (CSV) ---
methods_csv = EXPORT_DIR / "fem_comparison_kim2024B.csv"
with open(methods_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["metodo", "T_max_C", "error_vs_paper_K"])
    w.writerow(["FEM_paper_COMSOL",          70.60,  0.00])
    w.writerow(["FEM_FEniCSx_malla_completa", T_max,  round(T_max - 70.6, 4)])
    w.writerow(["PINN_64x4_Kennelly",         70.19,  round(70.19 - 70.6, 4)])
    w.writerow(["PINN_128x5_destilado",        70.17,  round(70.17 - 70.6, 4)])
print(f"✓ Comparación métodos: {methods_csv}")

print()
print("Archivos en Drive:")
for p in sorted(EXPORT_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<45}  {size_kb:>8.1f} KB")


## Celda 9 — Exportar malla final a Drive (formato XDMF para DOLFINx)
Guarda la malla FEniCSx con sus etiquetas de material y frontera en formato XDMF/HDF5.
Se puede recargar directamente en DOLFINx sin reconstruir con Gmsh.

In [ ]:
from dolfinx.io import XDMFFile

MESH_DIR = Path("/content/drive/MyDrive/Fenics_tesis")
MESH_DIR.mkdir(parents=True, exist_ok=True)

# --- Malla + cell_tags (etiquetas de material) ---
mesh_xdmf = str(MESH_DIR / "mesh_kim2024B.xdmf")
with XDMFFile(MPI.COMM_WORLD, mesh_xdmf, "w") as xdmf:
    xdmf.write_mesh(msh)
    xdmf.write_meshtags(cell_tags, msh.geometry)
print(f"✓ Malla + cell_tags: {mesh_xdmf}")
print(f"  (.h5 generado junto al .xdmf)")

# --- facet_tags (etiquetas de frontera: top=10, bot=11, left=12, right=13) ---
msh.topology.create_connectivity(1, 2)   # requerido para write_meshtags 1D
facet_xdmf = str(MESH_DIR / "facet_tags_kim2024B.xdmf")
with XDMFFile(MPI.COMM_WORLD, facet_xdmf, "w") as xdmf:
    xdmf.write_mesh(msh)
    xdmf.write_meshtags(facet_tags, msh.geometry)
print(f"✓ facet_tags:        {facet_xdmf}")

# --- Solución T_fem ---
sol_xdmf = str(MESH_DIR / "T_fem_kim2024B.xdmf")
with XDMFFile(MPI.COMM_WORLD, sol_xdmf, "w") as xdmf:
    xdmf.write_mesh(msh)
    T_fem.name = "T_K"
    xdmf.write_function(T_fem)
print(f"✓ Solución T_fem:    {sol_xdmf}")

print()
print("─" * 55)
print("Para recargar en DOLFINx (sin reconstruir con Gmsh):")
print()
print("  from dolfinx.io import XDMFFile")
print("  from mpi4py import MPI")
print()
print("  with XDMFFile(MPI.COMM_WORLD, 'mesh_kim2024B.xdmf', 'r') as f:")
print("      msh       = f.read_mesh(name='mesh')")
print("      cell_tags = f.read_meshtags(msh, name='cell_tags')")
print()
print("  with XDMFFile(MPI.COMM_WORLD, 'facet_tags_kim2024B.xdmf', 'r') as f:")
print("      f.read_mesh(name='mesh')  # necesario primero")
print("      facet_tags = f.read_meshtags(msh, name='facet_tags')")
print("─" * 55)

print()
print("Archivos de malla en Drive:")
for p in sorted(MESH_DIR.glob("*.xdmf")) :
    h5 = p.with_suffix(".h5")
    size_xdmf = p.stat().st_size / 1024
    size_h5   = h5.stat().st_size / 1024 if h5.exists() else 0
    print(f"  {p.name:<40}  {size_xdmf:>7.1f} KB  (+h5: {size_h5:>8.1f} KB)")
